# Análisis Exploratorio y Calidad de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

## 1. Carga de Datos Crudos (Raw Data)
Cargamos los archivos CSV principales que contienen la información sobre los servicios del hogar y las características de la vivienda.

## 2. Deteccion de Duplicados
Verificamos la integridad de las llaves primarias `DIRECTORIO` (Identificador del hogar) y `SECUENCIA_P` (Identificador de la persona).

In [ ]:
# Revisar duplicados por llaves primarias
dups_serv = df_servicios.duplicated(subset=['DIRECTORIO', 'SECUENCIA_P']).sum()
dups_viv = df_vivienda.duplicated(subset=['DIRECTORIO', 'SECUENCIA_P']).sum()

print(f"Duplicados en Servicios del Hogar: {dups_serv}")
print(f"Duplicados en Datos de la Vivienda: {dups_viv}")

# Acción recomendada: Eliminar los 212 registros duplicados en Servicios del Hogar manteniendo la primera ocurrencia.

## 3. Análisis de Valores Nulos y Atípicos en Variables Críticas
Analizaremos el costo de la energía eléctrica (`P5018`) y la estratificación (`P5010`), ya que son la base para calcular la huella de carbono eléctrica.

In [ ]:
# Análisis de la variable P5018 (Costo energía eléctrica)
nulos_energia = df_servicios['P5018'].isnull().sum()
ceros_energia = (df_servicios['P5018'] == 0).sum()
negativos_energia = (df_servicios['P5018'] < 0).sum()

print(f"Nulos en Pago Energía (P5018): {nulos_energia} ({(nulos_energia/len(df_servicios))*100:.2f}%)")
print(f"Ceros en Pago Energía (P5018): {ceros_energia}")
print(f"Negativos en Pago Energía (P5018): {negativos_energia}")

# Visualización de la distribución de pagos
plt.figure(figsize=(10, 5))
sns.boxplot(x=df_servicios['P5018'])
plt.title('Distribución de Pagos de Energía Eléctrica (Outliers)')
plt.show()

# Detectar atípicos usando el percentil 99
p99 = df_servicios['P5018'].quantile(0.99)
outliers = (df_servicios['P5018'] > p99).sum()
print(f"Valores por encima del percentil 99 (>{p99}): {outliers}")

In [ ]:
# Análisis de la variable P5046 (Gasto Gas Natural)
if 'P5046S1A1' in df_servicios.columns:
    display(df_servicios['P5046S1A1'].describe())

# Análisis de la variable P5067 (Gasto Gas Cilindro)
if 'P5067' in df_servicios.columns:
    display(df_servicios['P5067'].describe())

## 4. Estrategia Formal de Preparación de Datos (Data Cleansing)
Con base en los hallazgos anteriores, se definen las siguientes reglas de limpieza para la Fase de Transformacion en el ETL:
1.  **Deduplicación:** Eliminar registros duplicados en `df_servicios` usando `DIRECTORIO` y `SECUENCIA_P`.
2.  **Manejo de Nulos (Imputación):** Los 20,078 valores nulos en el costo de energía (`P5018`) se imputarán utilizando la **mediana del pago según el estrato socioeconómico (`P5010`)**.
3.  **Filtrado de Atípicos (Outliers):** Los registros con pagos de energía atípicamente altos (por encima del percentil 99, ej. > $450,000) deberán ser truncados o evaluados con cautela para no distorsionar la huella `romedio.
4.  **Limpieza de Ceros:** Los 250 registros con pago en $0 deberán ser analizados; si pertenecen a estrato 1 (donde el subsidio cubre todo), se considera válido. Si no, se imputarán con la mediana de su estrato.